# Prolog Program Explainer using LangChain

This notebook creates a tool that provides natural language explanations of Prolog programs using LangChain.

In [1]:
# Import necessary libraries
import os
from dotenv import load_dotenv
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List
from langchain_community.llms import HuggingFaceHub
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain.output_parsers import PydanticOutputParser

# Load environment variables (especially API keys)
load_dotenv()

True

In [2]:
# Define the output structure
class PrologExplanation(BaseModel):
    summary: str = Field(description="A brief summary of what the Prolog program does")
    predicates: List[dict] = Field(description="List of predicates and their explanations")
    logic_explanation: str = Field(description="How the program logic works")
    examples: List[str] = Field(description="Examples of how the program might be queried and the results")

In [3]:
# Define model options
def get_model(model_choice="deepseek"):
    if model_choice == "deepseek":
        # DeepSeek through Hugging Face
        return HuggingFaceHub(
            repo_id="deepseek-ai/deepseek-coder-33b-instruct",
            model_kwargs={"temperature": 0.1, "max_length": 2048}
        )
    elif model_choice == "llama":
        # Llama through Hugging Face
        return HuggingFaceHub(
            repo_id="meta-llama/Llama-2-13b-chat-hf",
            model_kwargs={"temperature": 0.1, "max_length": 2048}
        )
    else:
        raise ValueError(f"Unsupported model: {model_choice}")

# Create an instance of the output parser
parser = PydanticOutputParser(pydantic_object=PrologExplanation)

# Set up the LLM
model = get_model(model_choice="llama")

# Define the explanation prompt
explanation_template = """
You are a helpful assistant that explains Prolog programs in natural language. 
Given the following Prolog code, explain how it works, what it does, and how it can be used.

Prolog code:
```prolog
{prolog_code}
```

Provide your explanation in a structured format following these guidelines:
{format_instructions}
"""

prompt = PromptTemplate(
    template=explanation_template,
    input_variables=["prolog_code"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# Create the chain
explanation_chain = LLMChain(llm=model, prompt=prompt, output_parser=parser)

/tmp/ipykernel_1412913/278661156.py:11: LangChainDeprecationWarning: The class `HuggingFaceHub` was deprecated in LangChain 0.0.21 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEndpoint``.
  return HuggingFaceHub(
/home/lun/anaconda3/envs/usml/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_1412913/278661156.py:45: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  explanation_chain = LLMChain(llm=model, prompt=prompt, output_parser=parser)


In [4]:
def explain_prolog(prolog_code):
    """Generate a natural language explanation of a Prolog program.
    
    Args:
        prolog_code (str): The Prolog code to explain
        
    Returns:
        PrologExplanation: Structured explanation of the Prolog program
    """
    result = explanation_chain.invoke({"prolog_code": prolog_code})
    return result

## Example Usage

In [5]:
# Example Prolog code - Family relationships
family_prolog = """
% Facts
parent(john, bob).
parent(john, mary).
parent(jane, bob).
parent(jane, mary).
parent(bob, ann).
parent(bob, pat).
parent(mary, jim).

% Rules
father(X, Y) :- parent(X, Y), male(X).
mother(X, Y) :- parent(X, Y), female(X).
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).
sibling(X, Y) :- parent(P, X), parent(P, Y), X \= Y.

% Gender facts
male(john).
male(bob).
male(jim).
female(jane).
female(mary).
female(ann).
female(pat).
"""

explanation = explain_prolog(family_prolog)
print(f"Summary: {explanation.summary}\n")
print("Predicates:")
for pred in explanation.predicates:
    print(f"- {pred.get('name')}: {pred.get('description')}")
print(f"\nLogic: {explanation.logic_explanation}\n")
print("Examples:")
for example in explanation.examples:
    print(f"- {example}")

<>:2: SyntaxWarning: invalid escape sequence '\='
<>:2: SyntaxWarning: invalid escape sequence '\='
/tmp/ipykernel_1412913/3177931759.py:2: SyntaxWarning: invalid escape sequence '\='
  family_prolog = """
/home/lun/anaconda3/envs/usml/lib/python3.13/site-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)
/tmp/ipykernel_1412913/3177931759.py:2: SyntaxWarning: invalid escape sequence '\='
  family_prolog = """


BadRequestError: (Request ID: Root=1-67e48d7f-099a708b40bd86c97f90e4ed;e79b0b5b-6b73-4114-8800-1e602019fe8b)

Bad request:
Model requires a Pro subscription; check out hf.co/pricing to learn more. Make sure to include your HF token in your query.

In [ ]:
# Example Prolog code - List processing
list_prolog = """
% Append two lists
append([], L, L).
append([H|T], L, [H|R]) :- append(T, L, R).

% Check if element is a member of a list
member(X, [X|_]).
member(X, [_|T]) :- member(X, T).

% Get the length of a list
list_length([], 0).
list_length([_|T], N) :- list_length(T, N1), N is N1 + 1.

% Reverse a list
reverse(L, R) :- reverse_acc(L, [], R).
reverse_acc([], Acc, Acc).
reverse_acc([H|T], Acc, R) :- reverse_acc(T, [H|Acc], R).
"""

explanation = explain_prolog(list_prolog)
print(f"Summary: {explanation.summary}\n")
print("Predicates:")
for pred in explanation.predicates:
    print(f"- {pred.get('name')}: {pred.get('description')}")
print(f"\nLogic: {explanation.logic_explanation}\n")
print("Examples:")
for example in explanation.examples:
    print(f"- {example}")

## Custom Prolog Code Explanation

In [ ]:
# Enter your own Prolog code to explain
custom_prolog = """
% Enter your Prolog code here
"""

# Uncomment and run when you have code to explain
# explanation = explain_prolog(custom_prolog)
# print(f"Summary: {explanation.summary}\n")
# print("Predicates:")
# for pred in explanation.predicates:
#     print(f"- {pred.get('name')}: {pred.get('description')}")
# print(f"\nLogic: {explanation.logic_explanation}\n")
# print("Examples:")
# for example in explanation.examples:
#     print(f"- {example}")